# Classicmodels API paths (`main.py`)

This notebook calls the **Customers**, **Orders**, and **OrderDetails** routes from **`app/main.py`** over HTTP using the **`requests`** package.

**Prerequisite:** Run the API from the repository root, for example **`uvicorn app.main:app --reload --port 8000`**. The default base URL is **`http://127.0.0.1:8000`**; override with the environment variable **`API_BASE_URL`** if you use another host or port.

**Note:** Cells that **`POST`**, **`PUT`**, or **`DELETE`** modify values. Run create/update/delete cells once if changes aren't reverted in the cell properly, or restore the file from git if you repeat them.

In [1]:
import os

import requests

BASE_URL = os.environ.get("API_BASE_URL", "http://127.0.0.1:8000").rstrip("/")

try:
    _health = requests.get(f"{BASE_URL}/health", timeout=5)
    _health.raise_for_status()
except requests.RequestException as exc:
    raise RuntimeError(
        f"Cannot reach API at {BASE_URL!r}. From the repo root run e.g. "
        "`uvicorn app.main:app --reload --port 8000`, then rerun this cell "
        "(or set API_BASE_URL if the server uses another host/port)."
    ) from exc

# Constants for testing
EXISTING_CUST_ID = "103"  # Atelier graphique
EXISTING_ORDER_ID = "10100"
NEW_CUST_ID = 7777

## 1. Customers
Testing `GET /customers` with template filtering.

In [2]:
# Get first 3 customers
resp = requests.get(f"{BASE_URL}/customers")
print("Total Items:", len(resp.json()["items"]))
print("First Item:", resp.json()["items"][0]["customerName"])

# Filter by City
resp_filter = requests.get(f"{BASE_URL}/customers", params={"city": "NYC"})
print("Customers in NYC:", len(resp_filter.json()["items"]))

Total Items: 122
First Item: Atelier graphique
Customers in NYC: 5


## 2. Customer CRUD Lifecycle
Creates, Retrieves, Updates, and Deletes a test customer.

In [4]:
# 1. POST (Create)
payload = {
    "customerNumber": NEW_CUST_ID,
    "customerName": "Notebook Test Ltd",
    "contactLastName": "Jupyter",
    "contactFirstName": "User",
    "phone": "555-9999",
    "addressLine1": "101 Python Way",
    "city": "DataScience",
    "country": "USA"
}
post_resp = requests.post(f"{BASE_URL}/customers", json=payload)
print(f"POST Status: {post_resp.status_code}, ID Created: {post_resp.json()}")

# 2. PUT (Update)
update_payload = {"customerName": "Updated Via Notebook"}
put_resp = requests.put(f"{BASE_URL}/customers/{NEW_CUST_ID}", json=update_payload)
print(f"PUT Status: {put_resp.status_code}, Response: {put_resp.json()}")

# 3. GET (Verify)
get_resp = requests.get(f"{BASE_URL}/customers/{NEW_CUST_ID}")
print(f"Verified Name: {get_resp.json()['customerName']}")

# 4. DELETE (Cleanup)
del_resp = requests.delete(f"{BASE_URL}/customers/{NEW_CUST_ID}")
print(f"DELETE Status: {del_resp.status_code}, Deleted Count: {del_resp.json()['deleted']}")

POST Status: 201, ID Created: 7777
PUT Status: 200, Response: {'updated': 1}
Verified Name: Updated Via Notebook
DELETE Status: 200, Deleted Count: 1


## 3. Order Details (Composite Keys)
Testing routes that identify a single row using `/orders/{orderNumber}/orderdetails/{productCode}`.

In [5]:
# 1. Get all details for a specific order
resp = requests.get(f"{BASE_URL}/orders/{EXISTING_ORDER_ID}/orderdetails")
details = resp.json()["items"]
print(f"Order {EXISTING_ORDER_ID} has {len(details)} line items.")

# Pick a product code from the result for specific testing
target_product = details[0]["productCode"]
original_qty = details[0]["quantityOrdered"]

# 2. GET Specific Line Item
line_resp = requests.get(f"{BASE_URL}/orders/{EXISTING_ORDER_ID}/orderdetails/{target_product}")
print(f"Specific Line Found: {line_resp.json()['productCode']} | Qty: {line_resp.json()['quantityOrdered']}")

# 3. PUT (Update Quantity)
# Note: We must send the productCode in the JSON body because the Resource logic requires it
update_line = {"productCode": target_product, "quantityOrdered": original_qty + 5}
put_line_resp = requests.put(
    f"{BASE_URL}/orders/{EXISTING_ORDER_ID}/orderdetails/{target_product}", 
    json=update_line
)
print(f"Update Line Status: {put_line_resp.status_code}, Rows Affected: {put_line_resp.json()['updated']}")

# 4. Restore original quantity (Cleanup)
restore_line = {"productCode": target_product, "quantityOrdered": original_qty}
requests.put(f"{BASE_URL}/orders/{EXISTING_ORDER_ID}/orderdetails/{target_product}", json=restore_line)
print("Data restored to original state.")

Order 10100 has 4 line items.
Specific Line Found: S18_1749 | Qty: 30
Update Line Status: 200, Rows Affected: 1
Data restored to original state.


## 4. Error Handling
Verifying 404 responses for invalid IDs.

In [6]:
# Testing 404
resp = requests.get(f"{BASE_URL}/customers/9999999")
print(f"Invalid GET Status: {resp.status_code}")
print(f"Error Detail: {resp.json()['detail']}")

Invalid GET Status: 404
Error Detail: No customer with id '9999999'


## `DELETE /harry-potter/{character_id}`

Returns **`{"deleted": 0}`** or **`{"deleted": 1}`**.

In [ ]:
resp = requests.delete(f"{BASE_URL}/harry-potter/{new_id}", timeout=30)
assert resp.status_code == 200
assert resp.json()["deleted"] == 1

gone = requests.get(f"{BASE_URL}/harry-potter/{new_id}", timeout=30)
assert gone.status_code == 404
gone.json()